In [2]:
import degirum as dg,mytools
from custom_postprocessors.AgeGenderRecognition import AgeGenderRecognition

In [3]:
cloud_token = mytools.get_token() # get cloud API access token from env.ini file
zoo = dg.connect(dg.CLOUD,cloud_zoo_url,cloud_token)
model = zoo.load_model("age_gender_recognition_retail--62x62_float_openvino_cpu_1")
model.custom_postprocessor = AgeGenderRecognition
model.image_backend = 'pil'

In [4]:
class AgeGenderRecognition(dg.postprocessor.DetectionResults):
    def __init__(self, *args, **kwargs): 
        super().__init__(*args, **kwargs)
        new_inference_results=[]
        
        gender_prediction = self._inference_results[0]["data"]
        age_prediction = self._inference_results[1]["data"]
        
        if gender_prediction[0][0][0][0] > gender_prediction[0][1][0][0]:
            result = {"gender": "Female","age" : int(age_prediction[0][0][0][0] * 100)}
        else:
            result = {"gender": "Male","age" : int(age_prediction[0][0][0][0] * 100)}
            
        new_inference_results.append(result)
        self._inference_results = new_inference_results

In [7]:
res=model("../age_gender/UTKFace/19_0_0_20170103201406775.jpg.chip.jpg")
res.results

[{'gender': 'Male', 'age': 24}]